# 22.10 模型监控与漂移检测 / Model Monitoring & Drift Detection

**中文**:模型部署上线,**不是终点,而是风险的开始**。一个上线时准确率 0.92 的模型,三个月后可能悄悄掉到 0.75——而且**它不会报错、不会崩溃,只是预测慢慢变烂**。原因是:训练数据是"过去世界"的快照,而真实世界一直在变(用户行为变了、疫情来了、竞品出现了、季节变了)。这叫**漂移(drift)**。监控的可怕之处在于它的**静默性**:没有报警、没有异常日志,你可能几周后才从下滑的业务指标里察觉。所以生产 ML 必须**主动监控**:输入数据变了吗?预测分布变了吗?模型还准吗?本节从零实现工业界最常用的两个漂移检测方法——**PSI(群体稳定性指数)** 和 **KS 检验**,并讲清数据漂移 vs 概念漂移的关键区别。这是 ML 从"部署"到"可靠运营"的最后一环,也是闭环(→触发重训)的起点。
**English**: Deploying a model **isn't the finish line but the start of risk**. A model launched at 0.92 accuracy may quietly drop to 0.75 three months later — and **it won't error, won't crash, just slowly gets worse**. Because training data is a snapshot of the "past world," while the real world keeps changing (user behavior shifts, a pandemic hits, a competitor appears, seasons change). This is **drift**. Monitoring's scary part is its **silence**: no alarms, no error logs, and you might notice only weeks later from declining business metrics. So production ML must **actively monitor**: did the input data change? did the prediction distribution change? is the model still accurate? This section implements industry's two most common drift-detection methods from scratch — **PSI (Population Stability Index)** and the **KS test** — and clarifies the key distinction between data drift and concept drift. This is ML's final link from "deployed" to "reliably operated," and the start of the loop (→ trigger retraining).

---

**中文**:**三种漂移(面试要能区分)**:
**English**: **Three kinds of drift (know the distinction for interviews)**:
- **中文**:**数据漂移 / 协变量漂移(data / covariate drift)**:**输入分布 $P(X)$ 变了**(比如用户平均年龄从 30 变成 40)。模型没变、世界的输入变了。用 **PSI / KS** 检测——不需要标签,能**实时**发现。
  **Data / covariate drift**: **the input distribution $P(X)$ changed** (e.g. users' average age went from 30 to 40). The model is unchanged, but the world's inputs shifted. Detect with **PSI / KS** — no labels needed, detectable **in real time**.
- **中文**:**概念漂移(concept drift)**:**输入和输出的关系 $P(y|X)$ 变了**(同样的用户特征,现在对应不同的行为——比如疫情让"高收入"不再预示"爱旅游")。这是最危险的,因为**输入分布可能看起来没变**,但模型的假设已经过时。只能靠**性能监控**(准确率下降)发现,而这需要真实标签。
  **Concept drift**: **the relationship $P(y|X)$ between input and output changed** (the same user features now correspond to different behavior — e.g. a pandemic makes "high income" no longer predict "loves travel"). The most dangerous, because **the input distribution may look unchanged** while the model's assumptions are outdated. Detectable only via **performance monitoring** (accuracy drop), which needs true labels.
- **中文**:**预测漂移(prediction drift)**:**模型输出分布变了**(比如"预测为欺诈"的比例突然翻倍)。是输入漂移的下游信号,也可能预示问题。

  **Prediction drift**: **the model's output distribution changed** (e.g. the fraction "predicted fraud" suddenly doubles). A downstream signal of input drift, and may foreshadow problems.

**中文**:**PSI(Population Stability Index)** 是最常用的数据漂移度量:把参考分布(训练时)和当前分布(线上)各自分箱,比较每个箱的占比:
**English**: **PSI (Population Stability Index)** is the most common data-drift metric: bin the reference (training-time) and current (online) distributions, and compare each bin's proportion:
$$\text{PSI}=\sum_{i}(p_i^{\text{prod}}-p_i^{\text{ref}})\cdot\ln\frac{p_i^{\text{prod}}}{p_i^{\text{ref}}}$$
**中文**:经验阈值:**PSI < 0.1 稳定,0.1–0.2 需关注,> 0.2 显著漂移**。**KS 检验**则比较两个分布的累积分布函数差异,给出统计显著性。
**English**: Rules of thumb: **PSI < 0.1 stable, 0.1–0.2 watch, > 0.2 significant drift**. The **KS test** compares the two distributions' CDFs and gives statistical significance.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 生产 ML 必考）**
> **中文**:**模型会静默退化**(不报错, 预测慢慢变烂)→必须主动监控。**三种漂移**:①**数据/协变量漂移**(P(X)变, 用 **PSI/KS** 检测, 无需标签可实时)②**概念漂移**(P(y|X)变, 最危险, 靠**性能监控**发现, 需标签)③**预测漂移**(输出分布变)。**PSI**=Σ(p_prod−p_ref)·ln(p_prod/p_ref), 阈值 <0.1 稳/0.1-0.2 关注/>0.2 漂移; **KS 检验**比 CDF 给显著性。**监控什么**:输入特征漂移、预测分布、**模型性能**(有标签时)、数据质量(缺失/异常/schema)、延迟/吞吐。**核心难题=标签延迟**:概念漂移要等真实标签(可能几天几周后才有), 所以先用无需标签的输入/预测漂移做早期预警。**漂移→行动**:告警→分析→**重训(接 22.7 持续训练)**或回滚。**工具**:**Evidently**(开源漂移报告)、WhyLabs、Arize、NannyML、云监控。面试金句:*"模型上线会因数据漂移(P(X)变)和概念漂移(P(y|X)变)静默退化; 用 PSI(>0.2 漂移)和 KS 检验监控输入分布(无需标签实时预警)、监控预测分布, 有标签时监控性能; 概念漂移最危险且受标签延迟困扰; 检测到漂移就告警并触发重训或回滚, 形成 MLOps 闭环; 工具 Evidently/Arize。"*
> **English**: **Models degrade silently** (no errors, predictions slowly worsen) → must monitor actively. **Three drifts**: ① **data/covariate drift** (P(X) changes, detect with **PSI/KS**, no labels, real-time) ② **concept drift** (P(y|X) changes, most dangerous, detected via **performance monitoring**, needs labels) ③ **prediction drift** (output distribution changes). **PSI** = Σ(p_prod−p_ref)·ln(p_prod/p_ref), thresholds <0.1 stable / 0.1-0.2 watch / >0.2 drift; **KS test** compares CDFs for significance. **What to monitor**: input feature drift, prediction distribution, **model performance** (when labels exist), data quality (missing/anomaly/schema), latency/throughput. **Core difficulty = label delay**: concept drift needs true labels (may arrive days/weeks later), so use label-free input/prediction drift for early warning. **Drift → action**: alert → analyze → **retrain (ties to 22.7 continuous training)** or roll back. **Tools**: **Evidently** (open-source drift reports), WhyLabs, Arize, NannyML, cloud monitoring. Interview line: *"Deployed models degrade silently from data drift (P(X) changes) and concept drift (P(y|X) changes); use PSI (>0.2 = drift) and the KS test to monitor input distributions (label-free real-time warning) and prediction distributions, and performance when labels exist; concept drift is most dangerous and hampered by label delay; on detected drift, alert and trigger retraining or rollback, forming the MLOps loop; tools are Evidently/Arize."*


In [ ]:

# ============================================================
# 从零实现 PSI + KS 数据漂移检测 / PSI + KS data-drift detection from scratch
# 中文:参考分布=训练时的特征分布; 生产分布=线上的当前分布。PSI/KS 量化两者差异, 判断是否漂移(无需标签)。
# English: reference = the feature's training-time distribution; production = the current online distribution.
#      PSI/KS quantify the difference to flag drift (no labels needed).
# ============================================================
import numpy as np
from scipy import stats
np.random.seed(0)
reference=np.random.normal(50, 10, 10000)                  # 训练时某特征的分布 / a feature's distribution at training

def psi(ref, prod, bins=10):                               # Population Stability Index
    edges=np.percentile(ref, np.linspace(0,100,bins+1)); edges[0]=-np.inf; edges[-1]=np.inf  # 按参考分位数分箱 / bin by ref quantiles
    r=np.histogram(ref, edges)[0]/len(ref) + 1e-6          # 参考各箱占比 / ref bin proportions
    p=np.histogram(prod, edges)[0]/len(prod) + 1e-6        # 生产各箱占比 / prod bin proportions
    return np.sum((p-r)*np.log(p/r))                       # Σ (p−r)·ln(p/r)

scenarios=[("无漂移 no drift",       np.random.normal(50,10,5000)),
           ("轻微偏移 mild shift",    np.random.normal(53,10,5000)),
           ("大幅偏移 big shift",     np.random.normal(60,12,5000)),
           ("方差变化 variance change",np.random.normal(50,20,5000))]
print(f"{'场景 scenario':24}{'PSI':>8}{'判定':>12}{'KS_stat':>10}{'KS_pvalue':>12}")
results=[]
for name, prod in scenarios:
    ps=psi(reference, prod); ks_stat, ks_p=stats.ks_2samp(reference, prod)   # KS 检验 / KS test
    flag="🚨漂移" if ps>0.2 else ("⚠️关注" if ps>0.1 else "✓稳定")
    results.append((name, ps, ks_stat, ks_p, flag))
    print(f"{name:24}{ps:>8.3f}{flag:>11}{ks_stat:>10.3f}{ks_p:>12.1e}")
print("\nPSI 阈值:<0.1 稳定, 0.1~0.2 需关注, >0.2 显著漂移。注意方差变化 PSI 也高——PSI 对分布形状变化敏感")


In [ ]:

# ============================================================
# 概念漂移:输入分布没怎么变, 但模型准确率崩了 / concept drift: inputs look similar, but accuracy collapses
# 中文:概念漂移最阴险——P(y|X)变了(同样的输入现在对应不同的结果)。输入分布 PSI 可能很低(看起来没事),
#      但模型准确率暴跌。这说明:光监控输入漂移不够, 有标签时必须也监控性能。
# English: concept drift is the most insidious — P(y|X) changed (same inputs now map to different outcomes). Input-drift PSI
#      may be low (looks fine), yet accuracy collapses. So input-drift monitoring alone is insufficient; monitor performance too.
# ============================================================
from sklearn.linear_model import LogisticRegression
np.random.seed(1)
Xtr=np.random.randn(3000,2); ytr=(Xtr[:,0]+Xtr[:,1]>0).astype(int)         # 训练:y 取决于 x0+x1 / label rule at train
model=LogisticRegression().fit(Xtr,ytr)
# 生产数据:输入分布几乎一样, 但"世界的规则"变了(现在 y 取决于 x0−x1)/ production: same input dist, but the RULE flipped
Xprod=np.random.randn(3000,2); y_old_rule=(Xprod[:,0]+Xprod[:,1]>0).astype(int)
y_new_rule=(Xprod[:,0]-Xprod[:,1]>0).astype(int)                            # 概念漂移:关系变了 / P(y|X) changed
acc_before=(model.predict(Xprod)==y_old_rule).mean()                       # 若规则没变的准确率 / accuracy if rule unchanged
acc_after =(model.predict(Xprod)==y_new_rule).mean()                       # 规则变了之后的真实准确率 / actual accuracy after drift
psi_x0=psi(Xtr[:,0], Xprod[:,0])                                           # 输入 x0 的漂移 / input-drift of x0
print(f"输入特征 x0 的 PSI = {psi_x0:.3f}  → {'稳定(看起来没问题)' if psi_x0<0.1 else '漂移'}")
print(f"模型准确率(概念漂移前, 规则不变): {acc_before:.3f}")
print(f"模型准确率(概念漂移后, 规则已变): {acc_after:.3f}  ← 暴跌!")
print("\n关键:输入分布 PSI 很低(输入没漂移), 但准确率从 ~0.9 崩到 ~0.5——这就是概念漂移。")
print("→ 只监控输入漂移会漏掉它; 有真实标签时必须同时监控模型性能(但标签常延迟到达, 是核心难题)")


In [ ]:

# ============================================================
# 可视化:漂移检测 + 监控闭环 / drift detection + the monitoring loop
# ============================================================
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(14,5))
# ① 参考 vs 漂移分布 + PSI / reference vs drifted distributions
ax[0].hist(reference,bins=40,density=True,alpha=0.5,color="#4C72B0",label="参考(训练时)ref")
ax[0].hist(scenarios[2][1],bins=40,density=True,alpha=0.5,color="#C44E52",label=f"生产(大幅偏移)PSI={results[2][1]:.2f}")
ax[0].hist(scenarios[0][1],bins=40,density=True,alpha=0.3,color="#55A868",label=f"生产(无漂移)PSI={results[0][1]:.2f}")
ax[0].set_title("数据漂移:参考 vs 生产分布"); ax[0].legend(fontsize=8); ax[0].set_xlabel("特征值")
# ② 各场景 PSI 条形 + 阈值 / PSI bars
names=[r[0].split()[0] for r in results]; psis=[r[1] for r in results]
cols=["#55A868" if p<0.1 else ("#DD8452" if p<0.2 else "#C44E52") for p in psis]
b=ax[1].bar(names,psis,color=cols)
for bar,p in zip(b,psis): ax[1].text(bar.get_x()+bar.get_width()/2,p+0.01,f"{p:.2f}",ha="center",fontsize=10,weight="bold")
ax[1].axhline(0.1,ls="--",color="#DD8452",alpha=0.7,label="0.1 关注线")
ax[1].axhline(0.2,ls="--",color="#C44E52",alpha=0.7,label="0.2 漂移线")
ax[1].set_ylabel("PSI"); ax[1].set_title("PSI 判定漂移(绿稳定/橙关注/红漂移)"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("/tmp/mlops10_viz.png",dpi=80); plt.show()
print("监控闭环:检测漂移(PSI/KS/性能)→告警→分析原因→触发重训(22.7 持续训练)或回滚→重新上线监控")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **模型的"静默退化"是生产 ML 最危险的失败模式**:软件出 bug 会报错、会崩溃、会有异常日志,你立刻知道。但模型退化**悄无声息**——它照常返回预测,只是预测越来越差。上线时 0.92,几个月后可能变 0.75,而系统监控一切"正常"(没有 500 错误、延迟也正常)。等你从下滑的转化率、上升的投诉里察觉,业务可能已经损失了几周。**主动漂移监控就是给这个静默杀手装上警报器**:PSI/KS 让你在**输入数据一开始偏移**时就收到预警,而不是等到业务指标崩了才后知后觉。这是"部署了一个模型"和"运营一个可靠的 ML 系统"的本质区别。
2. **数据漂移能实时抓,概念漂移却受困于"标签延迟"——这是监控最深的痛点**:数据漂移(输入分布变)用 PSI/KS **不需要标签就能实时检测**——线上数据一进来就能算。但**概念漂移(输入-输出关系变)最危险却最难抓**:我们的实验里,输入分布的 PSI 很低(看起来一切正常),模型准确率却从 0.9 崩到 0.5——因为"世界的规则变了"。要发现它必须监控**真实性能**,而这需要**真实标签**。问题是:很多场景的标签**严重延迟**(信贷违约要等几个月才知道、用户会不会流失要观察很久)。这意味着当你终于拿到标签、算出"准确率崩了"时,烂模型可能已经在线上跑了很久。所以实践的智慧是:**用无需标签的输入漂移和预测漂移做"早期预警"**(虽然它们不能直接证明性能下降,但输入大幅漂移往往是性能下降的前兆),同时尽快回收标签做性能监控。
3. **诚实的复杂性:漂移检测容易,"漂移了该怎么办"才是真问题**。①**PSI/KS 会误报**:大数据量下 KS 检验对**任何**微小差异都显著(p 值极小),但那点差异可能业务上无所谓;PSI 阈值(0.1/0.2)也是经验值、不是金科玉律。所以检测到"漂移"不等于"必须行动"——要结合业务影响判断。②**漂移的根因分析很难**:是真实世界变了?还是上游数据管道坏了(某个特征突然全空)?还是节假日的正常波动?误把"数据管道 bug"当"概念漂移"去重训,只会训出更烂的模型。③**行动是权衡**:检测到漂移后,选项有——重训(接 22.7 持续训练,但要有新标签且走质量门)、回滚到旧模型、或仅告警观察。频繁重训有成本和风险,不重训则任由退化。④**监控本身要监控**:告警疲劳(alert fatigue)会让团队忽略真警报,阈值要调好。**结论:模型监控给静默退化的生产模型装上警报——用 PSI/KS 实时抓数据漂移、用性能监控抓概念漂移(受标签延迟所困);但检测只是开始, 真正的功力在于区分真漂移与误报/数据 bug、判断业务影响、并决定重训/回滚/观察——这构成了 MLOps 闭环(监控→重训→再部署→再监控),是 ML 系统长期可靠运营的核心。**

**English**:
1. **A model's "silent degradation" is production ML's most dangerous failure mode**: software bugs error, crash, log exceptions — you know instantly. But model degradation is **silent** — it returns predictions as usual, just increasingly bad. Launched at 0.92, months later maybe 0.75, while system monitoring says all "normal" (no 500 errors, latency fine). By the time you notice from declining conversion or rising complaints, the business may have lost weeks. **Active drift monitoring installs an alarm on this silent killer**: PSI/KS warn you **when input data first shifts**, not after business metrics collapse. This is the essential difference between "deployed a model" and "operating a reliable ML system."
2. **Data drift is caught in real time, but concept drift is trapped by "label delay" — monitoring's deepest pain**: data drift (input distribution change) is **detectable in real time without labels** via PSI/KS — computable the moment live data arrives. But **concept drift (input-output relationship change) is the most dangerous yet hardest to catch**: in our experiment, the input distribution's PSI was low (looks fine) while accuracy collapsed from 0.9 to 0.5 — because "the world's rule changed." Detecting it requires monitoring **actual performance**, which needs **true labels**. The problem: in many scenarios labels are **heavily delayed** (credit default known months later, churn observed over a long time). So by the time you finally get labels and compute "accuracy collapsed," the bad model may have run for a long time. Hence the practical wisdom: **use label-free input drift and prediction drift for "early warning"** (they don't directly prove performance decline, but large input drift often foreshadows it), while collecting labels ASAP for performance monitoring.
3. **Honest complexity: detecting drift is easy; "what to do about it" is the real problem**. ① **PSI/KS false-alarm**: at large data volumes the KS test is significant for **any** tiny difference (extremely small p-value), but that difference may be business-irrelevant; PSI thresholds (0.1/0.2) are rules of thumb, not gospel. So detecting "drift" ≠ "must act" — judge with business impact. ② **Root-cause analysis is hard**: did the real world change? or did the upstream data pipeline break (a feature suddenly all null)? or is it normal holiday fluctuation? Mistaking a "data pipeline bug" for "concept drift" and retraining only trains a worse model. ③ **Action is a tradeoff**: after detecting drift, options are — retrain (ties to 22.7 continuous training, but need new labels and a quality gate), roll back to the old model, or just alert and observe. Frequent retraining has cost and risk; not retraining lets degradation continue. ④ **Monitoring itself needs monitoring**: alert fatigue makes teams ignore real alarms, so tune thresholds well. **Conclusion: model monitoring installs an alarm on silently-degrading production models — PSI/KS catch data drift in real time, performance monitoring catches concept drift (trapped by label delay); but detection is only the start — the real skill is distinguishing true drift from false alarms/data bugs, judging business impact, and deciding retrain/rollback/observe — forming the MLOps loop (monitor→retrain→redeploy→monitor), the core of long-term reliable ML operation.**

> 💼 **实战视角 / Practical angle**
> **中文**:模型监控落地:①**监控四层**:数据质量(缺失/异常/schema, 接 22.7)、输入特征漂移(PSI/KS, 逐特征)、预测分布漂移、模型性能(有标签时);②**PSI 逐特征算**, >0.2 告警、0.1-0.2 关注; 大数据量时 KS 会过度显著, 结合效应量和业务判断;③**概念漂移**靠性能监控, 想办法**加速标签回收**(人工标注一小批、用代理指标), 用输入/预测漂移做早期预警;④**漂移→行动**:告警→根因分析(真漂移?数据 bug?季节性?)→重训(走 22.7 质量门)/回滚/观察;⑤**工具**:**Evidently**(开源, 自动漂移报告)、Arize/WhyLabs/NannyML、云监控 + Grafana 看板;⑥**闭环**:监控触发持续训练(22.7)。**别做的**:上线后不监控、把 KS 显著当必须重训、把数据管道 bug 当概念漂移。面试金句:*"生产模型会静默退化, 要监控数据质量、输入漂移(PSI>0.2 告警/KS)、预测漂移、性能; 数据漂移无需标签可实时检测, 概念漂移最危险但受标签延迟困扰所以先用输入漂移早期预警; 检测到漂移要先做根因分析(真漂移 vs 数据 bug)再决定重训/回滚, 形成监控→重训闭环; 工具用 Evidently/Arize。"*
> **English**: Model monitoring in practice: ① **four monitoring layers**: data quality (missing/anomaly/schema, ties to 22.7), input feature drift (PSI/KS, per feature), prediction distribution drift, model performance (when labels exist); ② **compute PSI per feature**, alert >0.2, watch 0.1-0.2; at large volumes KS over-triggers, so combine with effect size and business judgment; ③ **concept drift** relies on performance monitoring, so **speed up label collection** (manually label a small batch, use proxy metrics) and use input/prediction drift for early warning; ④ **drift → action**: alert → root-cause analysis (true drift? data bug? seasonality?) → retrain (through 22.7's quality gate) / rollback / observe; ⑤ **tools**: **Evidently** (open-source, auto drift reports), Arize/WhyLabs/NannyML, cloud monitoring + Grafana dashboards; ⑥ **loop**: monitoring triggers continuous training (22.7). **Don't**: skip monitoring after launch, treat KS significance as mandatory retraining, or mistake a data-pipeline bug for concept drift. Interview line: *"Production models degrade silently, so monitor data quality, input drift (PSI>0.2 alert/KS), prediction drift, and performance; data drift is label-free and real-time, concept drift is most dangerous but trapped by label delay so use input drift for early warning; on detected drift do root-cause analysis (true drift vs data bug) before deciding retrain/rollback, forming the monitor→retrain loop; tools are Evidently/Arize."*

---
### 小结 / Summary
- **中文**:模型上线会静默退化; 三种漂移:数据漂移(P(X)变, PSI/KS 实时无标签)、概念漂移(P(y|X)变, 靠性能监控, 受标签延迟困)、预测漂移。
- **English**: Deployed models degrade silently; three drifts: data drift (P(X) changes, PSI/KS real-time label-free), concept drift (P(y|X) changes, via performance monitoring, trapped by label delay), prediction drift.
- **中文**:PSI=Σ(p_prod−p_ref)·ln(p_prod/p_ref), <0.1 稳/0.1-0.2 关注/>0.2 漂移; 概念漂移最危险(输入看似没变但准确率崩)。
- **English**: PSI = Σ(p_prod−p_ref)·ln(p_prod/p_ref), <0.1 stable / 0.1-0.2 watch / >0.2 drift; concept drift most dangerous (inputs look unchanged but accuracy collapses).
- **中文**:检测只是开始——要区分真漂移与数据 bug、判业务影响、决定重训/回滚, 构成监控→重训闭环; 工具 Evidently/Arize。
- **English**: Detection is only the start — distinguish true drift from data bugs, judge business impact, decide retrain/rollback, forming the monitor→retrain loop; tools Evidently/Arize.
